In [2]:
import torch
import numpy as np
from torch.utils import data

In [3]:
def synthetic_data(w, b, num_examples):
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))

In [4]:
true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)

In [5]:
def load_array(data_arrays, batch_size, is_train=True):
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

batch_size = 10
data_iter = load_array((features, labels), batch_size)

In [6]:
from torch import nn

net = nn.Sequential(nn.Linear(2, 1))

In [8]:
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

In [9]:
loss = nn.MSELoss()

In [10]:
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

In [11]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X), y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch+1}, loss {l:f}')

epoch 1, loss 0.000165
epoch 2, loss 0.000101
epoch 3, loss 0.000100


In [13]:
w = net[0].weight.data
print(f'true_w - w: {true_w - w.reshape(true_w.shape)}')
b = net[0].bias.data
print(f'true_b - b: {true_b - b}')

true_w - w: tensor([-0.0002, -0.0005])
true_b - b: tensor([-0.0002])


# Exercise

## Task A

In [48]:
import torch
from torch import nn
from torch.utils import data


In [49]:
true_w = torch.tensor([-4.0])
true_b = torch.tensor([8.0])

X = torch.randn(500, 1)
y = torch.matmul(X, true_w) + true_b
y += torch.normal(0, 0.01, y.shape)

features, labels = X, y

In [ ]:
batch_size = 10

def load_data(data_array, batch_size, is_train=True):
    dataset = data.TensorDataset(*data_array)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

# better to build the iter once, outside the training loop
train_iter = load_data((features, labels), batch_size)

In [ ]:
net = nn.Sequential(nn.Linear(1, 1))
loss = nn.MSELoss()
optimizer = torch.optim.SGD(net.parameters(), lr=0.03)

epochs = 3

for epoch in range(epochs):
    # for e_X, e_y in load_data((features, labels), batch_size):
    for e_X, e_y in train_iter:
        y_hat = net(e_X)
        l = loss(y_hat, e_y.reshape(y_hat.shape))
        optimizer.zero_grad()
        l.backward()
        optimizer.step()
    pred = net(features)
    total_loss = loss(pred, labels.reshape(pred.shape))
    print(f'epoch {epoch+1}, loss {total_loss:f}')

epoch 1, loss 0.198838
epoch 2, loss 0.000603
epoch 3, loss 0.000104


In [50]:
print(net[0].weight.data)
print(net[0].bias.data)

tensor([[-3.9990]])
tensor([8.0003])


In [55]:
torch.allclose(true_w.reshape_as(net[0].weight), net[0].weight.detach())


False

In [56]:
torch.allclose(true_b, net[0].bias.data)

False

In [58]:
# 修正
w_hat = net[0].weight.detach().reshape(-1)
b_hat = net[0].bias.detach().reshape(-1)

assert torch.abs(w_hat - true_w).item() < 0.01, 'w failed to learn'
assert torch.abs(b_hat - true_b).item() < 0.05, 'b failed to learn'
print('assert passed')

assert passed


## Task B

In [26]:
import torch
from torch import nn, optim
import math
import wandb
import uuid

In [27]:
X = torch.linspace(-math.pi, math.pi, 1000).reshape(-1,1)
y = torch.sin(X).reshape(-1,1)

In [28]:
features, labels = X, y

In [29]:
# Run 1
net1 = nn.Linear(1, 1)
optimizer = optim.SGD(net1.parameters(), lr=0.01)
loss = nn.MSELoss()

if wandb.run is not None:
    wandb.finish() 
run_id = uuid.uuid4().hex[:8]
wandb.init(project="torchlearn-stage1", name=f"day07-nn-rewrite-run1-{run_id}")

for epoch in range(500):
    y_hat = net1(features).reshape(-1, 1)
    l = loss(y_hat, labels)
    optimizer.zero_grad()
    l.backward()
    optimizer.step()

    wandb.log({"loss": l.item()})
wandb.finish()

loss,▇▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▆█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss,0.00035


loss,█▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss,0.19675


In [30]:
# Run 2
net2 = nn.Sequential(
    nn.Linear(1, 64),
    nn.Tanh(),
    nn.Linear(64, 1)
)
optimizer = optim.Adam(net2.parameters(), lr=0.01)
loss = nn.MSELoss()

if wandb.run is not None:
    wandb.finish()
run_id = uuid.uuid4().hex[:8]
wandb.init(project="torchlearn-stage1", name=f"day07-nn-rewrite-run2-{run_id}")

for epoch in range(2000):
    y_hat = net2(features).reshape(-1, 1)
    l = loss(y_hat, labels)
    optimizer.zero_grad()
    l.backward()
    optimizer.step()

    wandb.log({"loss": l.item()})
wandb.finish()

loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss,0.0


Q1: 线性模型为什么拟合不了 sin？
A1: 单一的线性模型永远无法拟合出sin的曲线

Q2: 加隐藏层后发生了什么？
A2：nn.Linear(1,64)提供了64个线性神经元，nn.Tanh()把这64个线性神经元的输出范围压缩到(-1,1)，nn.Linear(64,1)把64个线性神经元的输出映射到一个范围，这个范围就是y的范围，即(-1,1)

Q3: 回归任务输出层为什么不加激活？
A3：因为回归的目标本身就没有上下限，而每一个激活函数都在偷偷给输出加一个上下限。加错了，模型就从结构上丧失了拟合能力。